# BiGRU + Conv1D + Dense — Unified Pipeline with GWO Optimization
## 2-Layer BiGRU · Conv1D · BatchNorm · Dense · Grey Wolf Optimizer

**Architecture:** `Embedding → Conv1D → BatchNorm → MaxPool → BiGRU×2 → Dense → Dropout → Output`  
**Optimization:** Grey Wolf Optimizer (GWO) via `mealpy` for automated hyperparameter tuning  

---
| Section | Content |
|---|---|
| 1 | Environment & Imports |
| 2 | Data Loading |
| 3 | Model Architecture |
| 4 | Metrics Utilities |
| 5 | Baseline Training |
| 6 | Evaluation & Visualisation |
| 7 | GWO Hyperparameter Optimisation |
| 8 | Final Model (Best Params) |
| 9 | Baseline vs Optimised Comparison |


## 1. Environment & Imports

In [45]:
# ============================================================
# 1.1  Environment Variables  (must be set BEFORE importing TF)
# ============================================================
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"]  = "2"    # Suppress TF info/warning logs
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"  # Use GPU 0 and 1 (set "0" for single GPU)
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # Disable OneDNN for numerical stability
os.environ["PYTHONHASHSEED"]        = "116"  # Hash seed for reproducibility

# ============================================================
# 1.2  Standard Library
# ============================================================
import warnings
import random
import time
from datetime import timedelta
from typing import Dict, Tuple, List, Optional

warnings.filterwarnings("ignore")

# ============================================================
# 1.3  Third-party
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1.4  TensorFlow / Keras
# ============================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Conv1D, BatchNormalization, MaxPooling1D,
    GRU, Bidirectional, Dense, Dropout , AdditiveAttention
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model

# ============================================================
# 1.5  Scikit-learn
# ============================================================
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, accuracy_score
)

# ============================================================
# 1.6  Mealpy (GWO)
# ============================================================
from mealpy.swarm_based.GWO import OriginalGWO
from mealpy import FloatVar, IntegerVar

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("✓ All imports successful")


TensorFlow : 2.15.0
NumPy      : 1.26.0
Pandas     : 2.3.3
✓ All imports successful


In [46]:
# ============================================================
# 1.7  Reproducibility
# ============================================================

RANDOM_STATE = 116

def set_seeds(seed: int = RANDOM_STATE) -> None:
    """Set all relevant random seeds for reproducibility."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

set_seeds(RANDOM_STATE)
print(f"✓ Seeds set (seed={RANDOM_STATE})")

# ============================================================
# 1.8  GPU Configuration
# ============================================================

def setup_gpu() -> None:
    """Enable dynamic memory growth to prevent GPU OOM errors."""
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        print("⚠  No GPU detected — running on CPU.")
        return
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical = tf.config.list_logical_devices("GPU")
        print(f"✓ GPUs: {len(gpus)} physical | {len(logical)} logical")
        for g in gpus:
            print(f"   {g.name}")
    except RuntimeError as e:
        print(f"⚠  GPU setup failed: {e}")

setup_gpu()


✓ Seeds set (seed=116)
✓ GPUs: 1 physical | 1 logical
   /physical_device:GPU:0


In [47]:
# ============================================================
# 1.9  Global Configuration
# ============================================================

# ── Dataset paths ───────────────────────────────────────────
DATA_PATHS: Dict[str, str] = {
    "train": "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/code/SMILES/train_set_balanced.xlsx",
    "val":   "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/code/SMILES/val_set_balanced.xlsx",
    "test":  "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/code/SMILES/test_set_balanced.xlsx",
}

# ── Training hyper-parameters ────────────────────────────────
EPOCHS   = 100
PATIENCE = 5

# ── Baseline (reference) hyper-parameters ───────────────────
# NOTE: These mirror the architecture fixed in Notebook A.
#       gru_units=64 → layer 1: 64 units; layer 2: 32 units (half)
#       dense_units=64 → head Dense layer
BASELINE_PARAMS: Dict[str, float] = {
    "learning_rate": 3e-4,
    "batch_size":    64,
    "gru_units":     64,    # first BiGRU layer; second uses gru_units // 2
    "cnn_filters":   64,
    "dense_units":   64,
    "dropout":       0.3,
}

# ── GWO configuration ────────────────────────────────────────
GWO_CONFIG: Dict[str, int] = {
    "pop_size": 10,   # number of wolves (population)
    "epoch":    10,   # number of GWO iterations
}

# ── Output directories ───────────────────────────────────────
MODEL_NAME  = "BiGRU_Dense_Conv"
DIRS = {
    "models" : "./models",
    "results": "./results",
    "plots"  : "./plots",
    "gwo"    : "./gwo_results",
    "images" : "./images",
}

for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

BASELINE_MODEL_PATH  = f"{DIRS['models']}/{MODEL_NAME}_baseline.keras"
OPTIMIZED_MODEL_PATH = f"{DIRS['models']}/{MODEL_NAME}_optimized.keras"
MODEL_ARCHITECTURE   = f"{DIRS['images']}/{MODEL_NAME}_architecture.png"

print("Configuration loaded.")
print(f"  Baseline model will save to : {BASELINE_MODEL_PATH}")
print(f"  Optimized model will save to: {OPTIMIZED_MODEL_PATH}")
print(f"  Model architecture image: {MODEL_ARCHITECTURE}")
print(f"  Epochs={EPOCHS} | Patience={PATIENCE} | Seed={RANDOM_STATE}")


Configuration loaded.
  Baseline model will save to : ./models/BiGRU_Dense_Conv_baseline.keras
  Optimized model will save to: ./models/BiGRU_Dense_Conv_optimized.keras
  Model architecture image: ./images/BiGRU_Dense_Conv_architecture.png
  Epochs=100 | Patience=5 | Seed=116


## 2. Data Loading

In [48]:
# SECTION 4 — DATA LOADING

def load_dataset(path: str, name: str) -> Tuple[np.ndarray, np.ndarray]:
    """Load dataset from Excel file and return features and labels as NumPy arrays."""
    
    df = pd.read_excel(path)

    X = df.drop(columns="labels").apply(pd.to_numeric, errors="coerce").fillna(0)
    y = pd.to_numeric(df["labels"], errors="coerce").fillna(0)

    X = np.asarray(X, dtype=np.int32)
    y = np.asarray(y, dtype=np.int32)

    label_counts = pd.Series(y).value_counts().rename(index={0: "Negative", 1: "Positive"})

    print(f"{name.upper():5s} | samples={X.shape[0]:,}  seq_len={X.shape[1]}  ", end="")
    print(f"dtype: X={X.dtype} y={y.dtype}")
    print(f"       label dist: {label_counts.to_dict()}")

    return X, y



# Load Datasets

print("LOADING DATASETS")


X_train, y_train = load_dataset(DATA_PATHS["train"], "train")
X_val,   y_val   = load_dataset(DATA_PATHS["val"],   "val")
X_test,  y_test  = load_dataset(DATA_PATHS["test"],  "test")


LOADING DATASETS
TRAIN | samples=8,260  seq_len=271  dtype: X=int32 y=int32
       label dist: {'Negative': 4167, 'Positive': 4093}
VAL   | samples=1,806  seq_len=271  dtype: X=int32 y=int32
       label dist: {'Positive': 913, 'Negative': 893}
TEST  | samples=1,833  seq_len=271  dtype: X=int32 y=int32
       label dist: {'Positive': 940, 'Negative': 893}


In [49]:

# Vocabulary Size & Sequence Length

VOCAB_SIZE = int(max(X_train.max(), X_val.max(), X_test.max()))
MAX_LEN    = X_train.shape[1]

print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"Training samples  : {X_train.shape[0]:,}")
print(f"Validation samples: {X_val.shape[0]:,}")
print(f"Test samples      : {X_test.shape[0]:,}")
print(f"Vocabulary size   : {VOCAB_SIZE:,}")
print(f"Sequence length   : {MAX_LEN}")
print(f"Positive (train)  : {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Positive (val)    : {y_val.sum():,}   ({y_val.mean()*100:.1f}%)")
print(f"Positive (test)   : {y_test.sum():,}  ({y_test.mean()*100:.1f}%)")
print("=" * 60)


DATASET STATISTICS
Training samples  : 8,260
Validation samples: 1,806
Test samples      : 1,833
Vocabulary size   : 132
Sequence length   : 271
Positive (train)  : 4,093 (49.6%)
Positive (val)    : 913   (50.6%)
Positive (test)   : 940  (51.3%)


## 3. Model Architecture

In [50]:
def build_model(
    vocab_size:         int,
    max_len:            int,
    gru_units:          int   = 128,
    cnn_filters:        int   = 64,
    dense_units:        int   = 64,
    dropout:            float = 0.3,
    cnn_kernel_size:    int   = 5,
    seed:               int   = RANDOM_STATE,
) -> Sequential:
    
    set_seeds(seed)

    model = Sequential([
        # ── Embedding ────────────────────────────────────────
        Embedding(
            input_dim    = vocab_size + 1,
            output_dim   = 128,
            input_length = max_len,
            mask_zero    = False,     
        ),

        # ── Convolutional block ──────────────────────────────
        Conv1D(filters=cnn_filters, kernel_size=cnn_kernel_size, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        # ── BiGRU layer 1 (wider, returns sequences) ─────────
        Bidirectional(GRU(gru_units,      return_sequences=True,  dropout=dropout)),

        # ── BiGRU layer 2 (narrower, returns last state) ─────
        Bidirectional(GRU(gru_units // 2, return_sequences=False, dropout=dropout)),

        
        # ── Dense head ───────────────────────────────────────
        Dense(dense_units, activation="relu"),
        
        Dropout(min(dropout + 0.1, 0.9)),   # slightly higher dropout in head

        # ── Output ───────────────────────────────────────────
        Dense(1, activation="sigmoid"),
    ], name="BiGRU_Dense_Conv")

    return model

# Convenience alias for GWO objective function (keeps naming consistent)
build_model = build_model


# ── Quick sanity check ────────────────────────────────────────────────────────
_m = build_model(VOCAB_SIZE, MAX_LEN)
_m.build(input_shape=(None, MAX_LEN))
_m.summary()

# Save architecture diagram
MODEL_ARCHITECTURE = os.path.join(DIRS["images"], f"{MODEL_NAME}_architecture.png")
plot_model(_m, to_file=MODEL_ARCHITECTURE, show_shapes=True, show_layer_names=True)
print(f"\n✓ Architecture diagram saved → {MODEL_ARCHITECTURE}")
del _m   # free memory; we'll build fresh before training


Model: "BiGRU_Dense_Conv"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_10 (Embedding)    (None, 271, 128)          17024     
                                                                 
 conv1d_10 (Conv1D)          (None, 271, 64)           41024     
                                                                 
 batch_normalization_10 (Ba  (None, 271, 64)           256       
 tchNormalization)                                               
                                                                 
 max_pooling1d_10 (MaxPooli  (None, 135, 64)           0         
 ng1D)                                                           
                                                                 
 bidirectional_20 (Bidirect  (None, 135, 256)          148992    
 ional)                                                          
                                                  

## 4. Metrics Utilities

In [51]:
class ClassificationMetrics:
    """
    Compute and store standard binary classification metrics.

    Usage
    -----
    cm = ClassificationMetrics(y_true, y_pred)
    print(cm.report())
    """

    def __init__(self, y_true: np.ndarray, y_pred: np.ndarray) -> None:
        self.y_true = y_true.flatten()
        self.y_pred = y_pred.flatten()
        self._cm    = confusion_matrix(self.y_true, self.y_pred)

    def confusion_matrix(self) -> np.ndarray:
        return self._cm

    def tp(self) -> int:
        return int(self._cm[1, 1])

    def tn(self) -> int:
        return int(self._cm[0, 0])

    def fp(self) -> int:
        return int(self._cm[0, 1])

    def fn(self) -> int:
        return int(self._cm[1, 0])

    def precision(self) -> float:
        return float(precision_score(self.y_true, self.y_pred, zero_division=0))

    def recall(self) -> float:
        return float(recall_score(self.y_true, self.y_pred, zero_division=0))

    def f1(self) -> float:
        return float(f1_score(self.y_true, self.y_pred, zero_division=0))

    def accuracy(self) -> float:
        return float(accuracy_score(self.y_true, self.y_pred))

    def as_dict(self) -> Dict:
        return {
            "Accuracy" : self.accuracy(),
            "Precision": self.precision(),
            "Recall"   : self.recall(),
            "F1-Score" : self.f1(),
            "TP": self.tp(), "TN": self.tn(),
            "FP": self.fp(), "FN": self.fn(),
        }

    def report(self, title: str = "Classification Metrics") -> str:
        d = self.as_dict()
        lines = ["=" * 50, title, "=" * 50]
        for k, v in d.items():
            lines.append(f"  {k:<12}: {v:.4f}" if isinstance(v, float) else f"  {k:<12}: {v}")
        lines.append("=" * 50)
        return "".join(lines)



# Utility

def to_numpy(x) -> np.ndarray:
    """Convert pandas DataFrame/Series to numpy array if needed."""    
    return x.values if hasattr(x, "values") else x


print("Helper classes/functions defined.")


Helper classes/functions defined.


## 5. Baseline Training

In [53]:
# ============================================================
# 5.1  Training Function
# ============================================================

set_seeds(RANDOM_STATE)
tf.config.optimizer.set_jit(False)   # Disable XLA/JIT (prevents libdevice errors)

print("=" * 60)
print("BASELINE MODEL TRAINING")
print("=" * 60)
print(f"  LR={BASELINE_PARAMS['learning_rate']}  Batch={BASELINE_PARAMS['batch_size']}  ", end="")
print(f"GRU={BASELINE_PARAMS['gru_units']}  Dropout={BASELINE_PARAMS['dropout']}")
print("=" * 60)

# ── Build ─────────────────────────────────────────────────
tf.keras.backend.clear_session()
set_seeds()

model = build_model(
    vocab_size  = VOCAB_SIZE,
    max_len     = MAX_LEN,
    gru_units   = int(BASELINE_PARAMS["gru_units"]),
    cnn_filters = int(BASELINE_PARAMS["cnn_filters"]),
    dense_units = int(BASELINE_PARAMS["dense_units"]),
    dropout     = float(BASELINE_PARAMS["dropout"]),
)
# ── Compile ───────────────────────────────────────────────
loss_fn = tf.keras.losses.BinaryFocalCrossentropy(gamma=2)
model.compile(
    optimizer = Adam(learning_rate=float(BASELINE_PARAMS["learning_rate"])),
    loss      = loss_fn,
    metrics   = ["accuracy", tf.keras.metrics.AUC(name="auc")],
)

    
model.build(input_shape=(None, MAX_LEN))
model.summary()

# ── Callbacks ─────────────────────────────────────────────
os.makedirs(os.path.dirname(BASELINE_MODEL_PATH) or ".", exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor             = "val_auc",
        mode                = "max",
        patience            = PATIENCE,
        restore_best_weights = True,
        verbose             = 1,
    ),
        
    ModelCheckpoint(
        filepath      = BASELINE_MODEL_PATH,   # .keras extension → native format
        monitor       = "val_auc",
        mode          = "max",
        save_best_only = True,
        verbose       = 1,
        ),
]

baseline_start = time.time()

# ── Train ─────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = EPOCHS,
    batch_size      = int(BASELINE_PARAMS["batch_size"]),
    callbacks       = callbacks,
    verbose         = 2,
)

elapsed = time.time() - baseline_start
print(f"\nBaseline training completed in {str(timedelta(seconds=int(elapsed)))} (elapsed: {elapsed:.2f} seconds)")




BASELINE MODEL TRAINING
  LR=0.0003  Batch=64  GRU=64  Dropout=0.3
Model: "BiGRU_Dense_Conv"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 271, 128)          17024     
                                                                 
 conv1d (Conv1D)             (None, 271, 64)           41024     
                                                                 
 batch_normalization (Batch  (None, 271, 64)           256       
 Normalization)                                                  
                                                                 
 max_pooling1d (MaxPooling1  (None, 135, 64)           0         
 D)                                                              
                                                                 
 bidirectional (Bidirection  (None, 135, 128)          49920     
 al)                                             

## 7. Grey Wolf Optimizer (GWO) Hyperparameter Search

In [54]:
# ============================================================
# 7.1  Search Space Definition
# ============================================================

# Variable order (must match objective_function extraction!):
#   [0] learning_rate   FloatVar
#   [1] batch_size      IntegerVar
#   [2] gru_units       IntegerVar
#   [3] dropout         FloatVar        ← NOTE: index 3, not cnn_filters
#   [4] cnn_filters     IntegerVar      ← NOTE: index 4
#   [5] dense_units     IntegerVar      ← NOTE: index 5

SEARCH_SPACE_INFO = {
    "learning_rate": (1e-5, 1e-2),
    "batch_size":    (32,   64),
    "gru_units":     (16,   128),
    "dropout":       (0.1,  0.5),
    "cnn_filters":   (16,   128),
    "dense_units":   (16,   128),
}

GWO_BOUNDS = [
    FloatVar   (lb=1e-5, ub=1e-2, name="learning_rate"),
    IntegerVar (lb=32,   ub=64,   name="batch_size"),
    IntegerVar (lb=16,   ub=128,  name="gru_units"),
    FloatVar   (lb=0.1,  ub=0.5,  name="dropout"),      # index 3
    IntegerVar (lb=16,   ub=128,  name="cnn_filters"),   # index 4
    IntegerVar (lb=16,   ub=128,  name="dense_units"),   # index 5
]


print("✓ GWO search space:")
for k, (lo, hi) in SEARCH_SPACE_INFO.items():
    print(f"  {k:<16}: [{lo}, {hi}]")


✓ GWO search space:
  learning_rate   : [1e-05, 0.01]
  batch_size      : [32, 64]
  gru_units       : [16, 128]
  dropout         : [0.1, 0.5]
  cnn_filters     : [16, 128]
  dense_units     : [16, 128]


In [55]:
# ============================================================
# 7.2  Objective Function
# ============================================================

# Tracking variables (reset before each GWO run)
_gwo_iter      = 0
_gwo_best_f1   = 0.0
_gwo_log: List[Dict] = []

def objective_function(solution: np.ndarray) -> float:
    """
    Objective for GWO minimisation.

    Extracts hyper-parameters from `solution` in the SAME order as
    GWO_BOUNDS, trains a lightweight model, and returns –F1 (min → max F1).

    Parameters
    ----------
    solution : np.ndarray
        [learning_rate, batch_size, gru_units, dropout, cnn_filters, dense_units]

    Returns
    -------
    float : –F1 score on the validation set (negative for minimisation).
    """
    global _gwo_iter, _gwo_best_f1

    _gwo_iter += 1

    # ── Extract parameters (ORDER MATCHES GWO_BOUNDS) ─────────
    learning_rate = float(solution[0])
    batch_size    = int(round(solution[1]))
    gru_units     = int(round(solution[2]))
    dropout       = float(solution[3])       # index 3
    cnn_filters   = int(round(solution[4]))  # index 4
    dense_units   = int(round(solution[5]))  # index 5  

    # ── Clamp to valid ranges ─────────────────────────────────
    batch_size  = max(32,  min(batch_size,  64))
    gru_units   = max(16,  min(gru_units,   128))
    cnn_filters = max(16,  min(cnn_filters, 128))
    dense_units = max(16,  min(dense_units, 128))
    dropout     = max(0.1, min(dropout,     0.5))

    print(f"\n[GWO iter {_gwo_iter:>3}]  "
          f"lr={learning_rate:.5f}  bs={batch_size}  "
          f"gru={gru_units}  cnn={cnn_filters}  "
          f"dense={dense_units}  drop={dropout:.3f}")

    try:
        tf.keras.backend.clear_session()
        set_seeds()

        model = build_model(
            vocab_size  = VOCAB_SIZE,
            max_len     = MAX_LEN,
            gru_units   = gru_units,
            cnn_filters = cnn_filters,
            dense_units = dense_units,
            dropout     = dropout,
        )

        loss_fn = tf.keras.losses.BinaryFocalCrossentropy(gamma=2)
        model.compile(
            optimizer = Adam(learning_rate=learning_rate),
            loss      = loss_fn,
            metrics   = ["accuracy", tf.keras.metrics.AUC(name="auc")],
        )

        early_stop = EarlyStopping(
            monitor             = "val_auc",
            mode                = "max",
            patience            = 3,
            restore_best_weights = True,
            verbose             = 0,
        )

        hist = model.fit(
            X_train, y_train,
            validation_data = (X_val, y_val),
            epochs          = 30,       # reduced for GWO efficiency
            batch_size      = batch_size,
            callbacks       = [early_stop],
            verbose         = 0,
        )

        y_prob = model.predict(X_val, verbose=0)
        y_pred = (y_prob > 0.5).astype(int).flatten()

        f1  = f1_score(y_val, y_pred, zero_division=0)
        acc = accuracy_score(y_val, y_pred)

        if f1 > _gwo_best_f1:
            _gwo_best_f1 = f1
            print(f"  ★ NEW BEST  F1={f1:.4f}  Acc={acc:.4f}")
        else:
            print(f"    F1={f1:.4f}  Acc={acc:.4f}")

        _gwo_log.append({
            "iteration":    _gwo_iter,
            "learning_rate": learning_rate,
            "batch_size":   batch_size,
            "gru_units":    gru_units,
            "cnn_filters":  cnn_filters,
            "dense_units":  dense_units,
            "dropout":      dropout,
            "f1_score":     f1,
            "accuracy":     acc,
            "epochs_run":   len(hist.history["loss"]),
        })

        return -f1     # GWO minimises → return negative F1

    except Exception as exc:
        print(f"  ✗ ERROR: {exc}")
        _gwo_log.append({
            "iteration": _gwo_iter,
            "f1_score": 0.0, "accuracy": 0.0,
            "learning_rate": learning_rate, "batch_size": batch_size,
            "gru_units": gru_units, "cnn_filters": cnn_filters,
            "dense_units": dense_units, "dropout": dropout,
            "epochs_run": 0,
        })
        return 0.0

print("✓ objective_function() defined")


✓ objective_function() defined


In [56]:
# ============================================================
# RUN GWO OPTIMISATION
# ============================================================

print("\n" + "=" * 60)
print("STARTING GWO OPTIMISATION")
print("=" * 60)
print(f"  Population size    : {GWO_CONFIG['pop_size']}")
print(f"  Epochs             : {GWO_CONFIG['epoch']}")
print(f"  Total evaluations  : {GWO_CONFIG['pop_size'] * GWO_CONFIG['epoch']}")
print("=" * 60 + "\n")

# Reset counters before each run (allows re-running this cell cleanly)
_iter_count  = 0
_best_f1_gwo = 0.0
_gwo_log     = []

gwo_optimizer = OriginalGWO(
    epoch    = GWO_CONFIG["epoch"],
    pop_size = GWO_CONFIG["pop_size"],
)

_gwo_start = time.time()
g_best = gwo_optimizer.solve({
    "obj_func": objective_function,
    "bounds":   GWO_BOUNDS,
    "minmax":   "min",
})
gwo_time = time.time() - _gwo_start

best_solution = g_best.solution
best_fitness  = g_best.target.fitness

print("\n" + "=" * 60)
print("GWO OPTIMISATION COMPLETE")
print("=" * 60)
print(f"  Total time  : {str(timedelta(seconds=int(gwo_time)))}")
print(f"  Best val F1 : {-best_fitness:.4f}")
print("=" * 60)

# ── Save optimisation log ─────────────────────────────────────────────────────
gwo_log_df = pd.DataFrame(_gwo_log)
_log_path  = os.path.join(DIRS["gwo"], f"{MODEL_NAME}_gwo_log.csv")
gwo_log_df.to_csv(_log_path, index=False)
print(f"✓ Optimisation log → {_log_path}")


2026/03/26 08:37:56 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: OriginalGWO(epoch=10, pop_size=10)



STARTING GWO OPTIMISATION
  Population size    : 10
  Epochs             : 10
  Total evaluations  : 100


[GWO iter   1]  lr=0.00726  bs=55  gru=63  cnn=39  dense=37  drop=0.230
  ★ NEW BEST  F1=0.9251  Acc=0.9291

[GWO iter   2]  lr=0.00156  bs=47  gru=67  cnn=72  dense=66  drop=0.139
  ★ NEW BEST  F1=0.9387  Acc=0.9385

[GWO iter   3]  lr=0.00008  bs=63  gru=94  cnn=43  dense=85  drop=0.137
  ★ NEW BEST  F1=0.9412  Acc=0.9430

[GWO iter   4]  lr=0.00201  bs=63  gru=69  cnn=43  dense=116  drop=0.273
    F1=0.9410  Acc=0.9408

[GWO iter   5]  lr=0.00514  bs=33  gru=69  cnn=30  dense=61  drop=0.245
    F1=0.9241  Acc=0.9286

[GWO iter   6]  lr=0.00498  bs=49  gru=111  cnn=95  dense=16  drop=0.400
    F1=0.9202  Acc=0.9252

[GWO iter   7]  lr=0.00470  bs=43  gru=122  cnn=18  dense=91  drop=0.309
    F1=0.9280  Acc=0.9319

[GWO iter   8]  lr=0.00900  bs=44  gru=85  cnn=107  dense=95  drop=0.317
    F1=0.9240  Acc=0.9280

[GWO iter   9]  lr=0.00320  bs=43  gru=117  cnn=84  dense=106  dro

2026/03/26 09:20:24 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 1, Current best: -0.9412435824301197, Global best: -0.9412435824301197, Runtime: 1620.37309 seconds


    F1=0.9380  Acc=0.9391

[GWO iter  21]  lr=0.00001  bs=46  gru=55  cnn=57  dense=113  drop=0.281
    F1=0.9212  Acc=0.9247

[GWO iter  22]  lr=0.00130  bs=64  gru=98  cnn=53  dense=81  drop=0.171
    F1=0.9332  Acc=0.9363

[GWO iter  23]  lr=0.00246  bs=64  gru=103  cnn=40  dense=128  drop=0.157
    F1=0.9408  Acc=0.9419

[GWO iter  24]  lr=0.00027  bs=32  gru=128  cnn=50  dense=99  drop=0.325
    F1=0.9308  Acc=0.9302

[GWO iter  25]  lr=0.00001  bs=50  gru=85  cnn=76  dense=109  drop=0.100
    F1=0.9274  Acc=0.9302

[GWO iter  26]  lr=0.00187  bs=64  gru=74  cnn=16  dense=128  drop=0.102
    F1=0.9392  Acc=0.9408

[GWO iter  27]  lr=0.00063  bs=64  gru=73  cnn=47  dense=128  drop=0.112
    F1=0.9346  Acc=0.9347

[GWO iter  28]  lr=0.00155  bs=64  gru=128  cnn=62  dense=103  drop=0.100
  ★ NEW BEST  F1=0.9420  Acc=0.9430

[GWO iter  29]  lr=0.00273  bs=64  gru=35  cnn=27  dense=128  drop=0.215
  ★ NEW BEST  F1=0.9429  Acc=0.9441

[GWO iter  30]  lr=0.00112  bs=59  gru=110  cnn=41  

2026/03/26 09:42:02 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 2, Current best: -0.9429055963821368, Global best: -0.9429055963821368, Runtime: 1342.30236 seconds


...

    F1=0.9375  Acc=0.9391

[GWO iter  91]  lr=0.00167  bs=64  gru=72  cnn=58  dense=79  drop=0.154
    F1=0.9302  Acc=0.9330

[GWO iter  92]  lr=0.00195  bs=62  gru=72  cnn=57  dense=83  drop=0.156
    F1=0.9218  Acc=0.9175

[GWO iter  93]  lr=0.00201  bs=64  gru=73  cnn=58  dense=79  drop=0.142
    F1=0.9292  Acc=0.9330

[GWO iter  94]  lr=0.00195  bs=64  gru=73  cnn=58  dense=77  drop=0.117
    F1=0.8985  Acc=0.8898

[GWO iter  95]  lr=0.00190  bs=60  gru=68  cnn=55  dense=82  drop=0.160
    F1=0.9352  Acc=0.9380

[GWO iter  96]  lr=0.00189  bs=64  gru=70  cnn=57  dense=83  drop=0.149
    F1=0.9352  Acc=0.9347

[GWO iter  97]  lr=0.00189  bs=64  gru=68  cnn=56  dense=83  drop=0.147
    F1=0.9332  Acc=0.9363

[GWO iter  98]  lr=0.00185  bs=59  gru=72  cnn=53  dense=86  drop=0.148
    F1=0.9332  Acc=0.9363

[GWO iter  99]  lr=0.00186  bs=64  gru=69  cnn=59  dense=87  drop=0.153
    F1=0.9352  Acc=0.9380

[GWO iter 100]  lr=0.00178  bs=58  gru=64  cnn=58  dense=81  drop=0.156


2026/03/26 11:26:04 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 9, Current best: -0.9433748584371461, Global best: -0.9433748584371461, Runtime: 805.99756 seconds


    F1=0.9342  Acc=0.9374

[GWO iter 101]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9336  Acc=0.9363

[GWO iter 102]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
  ★ NEW BEST  F1=0.9452  Acc=0.9463

[GWO iter 103]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9381  Acc=0.9391

[GWO iter 104]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9382  Acc=0.9385

[GWO iter 105]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9343  Acc=0.9369

[GWO iter 106]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9279  Acc=0.9319

[GWO iter 107]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9386  Acc=0.9385

[GWO iter 108]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9310  Acc=0.9341

[GWO iter 109]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148
    F1=0.9291  Acc=0.9330

[GWO iter 110]  lr=0.00181  bs=63  gru=69  cnn=57  dense=81  drop=0.148


2026/03/26 11:38:54 AM, INFO, mealpy.swarm_based.GWO.OriginalGWO: >>>Problem: P, Epoch: 10, Current best: -0.9452286843591191, Global best: -0.9452286843591191, Runtime: 769.80449 seconds


    F1=0.9429  Acc=0.9435

GWO OPTIMISATION COMPLETE
  Total time  : 3:00:57
  Best val F1 : 0.9452
✓ Optimisation log → ./gwo_results/BiGRU_Dense_Conv_gwo_log.csv


In [57]:
# ============================================================
# 7.4  Extract Best Hyperparameters  (FIXED indices)
# ============================================================
# FIX: Notebook B had a bug where solution[3] was mapped to
#      cnn_filters and solution[5] to dropout, reversing dropout
#      and cnn_filters.  Correct mapping shown below matches
#      GWO_BOUNDS definition exactly.

best_solution = g_best.solution

best_params: Dict = {
    "learning_rate": float(best_solution[0]),
    "batch_size":    int(round(best_solution[1])),
    "gru_units":     int(round(best_solution[2])),
    "dropout":       float(best_solution[3]),       # index 3 → dropout
    "cnn_filters":   int(round(best_solution[4])),  # index 4 → cnn_filters 
    "dense_units":   int(round(best_solution[5])),# index 5 → dense_units
}

print("=" * 60)
print("BEST HYPERPARAMETERS (GWO)")
print("=" * 60)
for k, v in best_params.items():
    baseline_v = BASELINE_PARAMS.get(k, "—")
    delta = ""
    if isinstance(v, float) and isinstance(baseline_v, float):
        delta = f"  (baseline: {baseline_v})"
    elif isinstance(baseline_v, (int, float)):
        delta = f"  (baseline: {baseline_v})"
    print(f"  {k:<16}: {v}{delta}")
print("=" * 60)

# Persist GWO log and best params
log_df = pd.DataFrame(_gwo_log)
log_df.to_csv(os.path.join(DIRS["gwo"], f"{MODEL_NAME}_gwo_log.csv"), index=False)
pd.DataFrame([best_params]).to_csv(
    os.path.join(DIRS["gwo"], f"{MODEL_NAME}_best_params.csv"), index=False
)
print(f"\n✓ GWO log saved        → {DIRS['gwo']}/{MODEL_NAME}_gwo_log.csv")
print(f"✓ Best params saved    → {DIRS['gwo']}/{MODEL_NAME}_best_params.csv")


BEST HYPERPARAMETERS (GWO)
  learning_rate   : 0.0018053180176007659  (baseline: 0.0003)
  batch_size      : 63  (baseline: 64)
  gru_units       : 69  (baseline: 64)
  dropout         : 0.14792322878061193  (baseline: 0.3)
  cnn_filters     : 57  (baseline: 64)
  dense_units     : 81  (baseline: 64)

✓ GWO log saved        → ./gwo_results/BiGRU_Dense_Conv_gwo_log.csv
✓ Best params saved    → ./gwo_results/BiGRU_Dense_Conv_best_params.csv


In [58]:

print("=" * 60)
print("FINAL MODEL TRAINING  (GWO-Optimised Hyperparameters)")
print("=" * 60)
print("Hyperparameters:", best_params)
print()

tf.keras.backend.clear_session()
set_seeds(RANDOM_STATE)

optimized_model   = build_model(
    vocab_size    = VOCAB_SIZE,
    max_len       = MAX_LEN,
    gru_units     = best_params["gru_units"],
    dense_units   = best_params["dense_units"],
    dropout       = best_params["dropout"],
    cnn_filters   = best_params["cnn_filters"],
)

optimized_model.compile(
    optimizer = Adam(learning_rate=best_params["learning_rate"]),
    loss      = tf.keras.losses.BinaryFocalCrossentropy(gamma=2),
    metrics   = ["accuracy", tf.keras.metrics.AUC(name="auc")],
)

optimized_model.summary()

optimized_save_path = os.path.join(DIRS["models"], f"{MODEL_NAME}_optimized_best.keras")

opt_start = time.time()
optimized_history = optimized_model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = EPOCHS,
    batch_size      = best_params["batch_size"],
    callbacks       = [
        EarlyStopping(monitor="val_auc", mode="max",
                      patience=PATIENCE, restore_best_weights=True, verbose=1),
        ModelCheckpoint(filepath=optimized_save_path, monitor="val_auc",
                        mode="max", save_best_only=True, verbose=1),
    ],
    verbose         = 2,
)

optimized_time = time.time() - opt_start

print(f"Optimized training completed in {str(timedelta(seconds=int(optimized_time)))}")
print(f"Best model saved to: {OPTIMIZED_MODEL_PATH}")


FINAL MODEL TRAINING  (GWO-Optimised Hyperparameters)
Hyperparameters: {'learning_rate': 0.0018053180176007659, 'batch_size': 63, 'gru_units': 69, 'dropout': 0.14792322878061193, 'cnn_filters': 57, 'dense_units': 81}

Model: "BiGRU_Dense_Conv"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 271, 128)          17024     
                                                                 
 conv1d (Conv1D)             (None, 271, 57)           36537     
                                                                 
 batch_normalization (Batch  (None, 271, 57)           228       
 Normalization)                                                  
                                                                 
 max_pooling1d (MaxPooling1  (None, 135, 57)           0         
 D)                                                              
                              

## 6. Baseline Evaluation & Visualisation

In [59]:

# Evaluate Both Models on Validation & Test Sets


def evaluate_model(model, X, y, label="") -> ClassificationMetrics:
    """Predict and return ClassificationMetrics for a given split."""    
    y_pred = (model.predict(X, verbose=0) > 0.5).astype(int).flatten()
    return ClassificationMetrics(y, y_pred)


# Baseline
baseline_val_metrics  = evaluate_model(baseline_model,  X_val,  y_val,  "Baseline Val")
baseline_test_metrics = evaluate_model(baseline_model,  X_test, y_test, "Baseline Test")

# Optimized
optimized_val_metrics  = evaluate_model(optimized_model, X_val,  y_val,  "Optimized Val")
optimized_test_metrics = evaluate_model(optimized_model, X_test, y_test, "Optimized Test")

# Print reports 
print(baseline_test_metrics.report("BASELINE — Test Set"))
print()
print(optimized_test_metrics.report("GWO-OPTIMIZED — Test Set"))


NameError: name 'baseline_model' is not defined

In [ ]:

# Side-by-Side Comparison DataFrame


def metrics_row(label, params, cm: ClassificationMetrics, train_time_s: float) -> Dict:
    d = cm.as_dict()
    return {
        "Model"         : label,
        "LR"            : params["learning_rate"],
        "Batch"         : params["batch_size"],
        "GRU_Units"     : params["gru_units"],
        "Dropout"       : params["dropout"],
        "Accuracy"      : d["Accuracy"],
        "Precision"     : d["Precision"],
        "Recall"        : d["Recall"],
        "F1-Score"      : d["F1-Score"],
        "TP"            : d["TP"],
        "TN"            : d["TN"],
        "FP"            : d["FP"],
        "FN"            : d["FN"],
        "Training_Time" : str(timedelta(seconds=int(train_time_s))),
    }


comparison_df = pd.DataFrame([
    metrics_row("Baseline",      BASELINE_PARAMS, baseline_test_metrics,  baseline_time),
    metrics_row("GWO-Optimized", best_params,     optimized_test_metrics, optimized_time),
])

f1_improvement  = (optimized_test_metrics.f1()  - baseline_test_metrics.f1())  / max(baseline_test_metrics.f1(),  1e-9) * 100
acc_improvement = (optimized_test_metrics.accuracy() - baseline_test_metrics.accuracy()) / max(baseline_test_metrics.accuracy(), 1e-9) * 100

print("" + "=" * 90)
print("BASELINE vs GWO-OPTIMIZED — TEST SET COMPARISON")
print("=" * 90)
print(comparison_df.to_string(index=False))
print("" + "=" * 90)
print(f"  F1 improvement  : {f1_improvement:+.2f}%")
print(f"  Acc improvement : {acc_improvement:+.2f}%")
print("=" * 90)

# Save comparison CSV
comp_path = f"{DIRS['results']}/{MODEL_NAME}_comparison.csv"
comparison_df.to_csv(comp_path, index=False)
print(f"Comparison saved to: {comp_path}")


In [ ]:

# Save Training History CSVs


def save_history_csv(history, name: str) -> str:
    epochs = len(history.history["loss"])
    df = pd.DataFrame({
        "epoch"         : range(1, epochs + 1),
        "train_loss"    : history.history["loss"],
        "val_loss"      : history.history["val_loss"],
        "train_accuracy": history.history["accuracy"],
        "val_accuracy"  : history.history["val_accuracy"],
        "train_auc"     : history.history["auc"],
        "val_auc"       : history.history["val_auc"],
    })
    path = f"{DIRS['results']}/{name}_history.csv"
    df.to_csv(path, index=False)
    print(f"  Saved: {path}")
    return path


print("Saving training histories...")
save_history_csv(history          ,   f"{MODEL_NAME}_baseline")
save_history_csv(optimized_history,  f"{MODEL_NAME}_optimized")


In [ ]:

# Training Curves — Baseline vs Optimized


fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric, title in zip(
    axes,
    [("accuracy", "val_accuracy"), ("auc", "val_auc")],
    ["Accuracy", "AUC"],
):
    train_key, val_key = metric
    ax.plot(history.history[val_key],
            label="Baseline Val",    linewidth=2, linestyle="--", color="#3498db")
    ax.plot(optimized_history.history[val_key],
            label="Optimized Val",   linewidth=2, color="#2ecc71")
    ax.set_title(f"Validation {title} Comparison", fontsize=14, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
train_cmp_path = f"{DIRS['plots']}/{MODEL_NAME}_training_comparison.png"
plt.savefig(train_cmp_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {train_cmp_path}")


## 8. Final Model Training (GWO-Optimised Hyperparameters)

In [ ]:

# Confusion Matrices — Baseline vs Optimized (Test Set)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cm_obj, title, cmap in zip(
    axes,
    [baseline_test_metrics, optimized_test_metrics],
    ["Baseline", "GWO-Optimized"],
    ["Blues", "Greens"],
):
    sns.heatmap(
        cm_obj.confusion_matrix(), annot=True, fmt="d",
        cmap=cmap, ax=ax,
        xticklabels=["Negative", "Positive"],
        yticklabels=["Negative", "Positive"],
        cbar_kws={"label": "Count"}, annot_kws={"size": 13},
    )
    ax.set_title(f"{title} — Confusion Matrix", fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)

plt.tight_layout()
cm_path = f"{DIRS['plots']}/{MODEL_NAME}_confusion_comparison.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")


In [ ]:

# Metrics Bar Chart — Test Set


metric_names  = ["Accuracy", "Precision", "Recall", "F1-Score"]
base_values   = [baseline_test_metrics.accuracy(), baseline_test_metrics.precision(),
                  baseline_test_metrics.recall(),   baseline_test_metrics.f1()]
opt_values    = [optimized_test_metrics.accuracy(), optimized_test_metrics.precision(),
                  optimized_test_metrics.recall(),   optimized_test_metrics.f1()]

x     = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, base_values, width, label="Baseline",      color="#3498db", alpha=0.85)
bars2 = ax.bar(x + width/2, opt_values,  width, label="GWO-Optimized", color="#2ecc71", alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.005,
                f"{h:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("Score", fontsize=12, fontweight="bold")
ax.set_title("Test Set Performance — Baseline vs GWO-Optimized", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim([0, 1.07])
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
bar_path = f"{DIRS['plots']}/{MODEL_NAME}_metrics_comparison.png"
plt.savefig(bar_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {bar_path}")


In [ ]:

# GWO Optimisation Progress


fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# F1 over iterations
axes[0].plot(gwo_log_df["iteration"], gwo_log_df["f1_score"],
             marker="o", linewidth=2, markersize=5, label="GWO Candidate")
axes[0].axhline(y=baseline_val_metrics.f1(), color="r", linestyle="--",
                label=f"Baseline Val F1 ({baseline_val_metrics.f1():.4f})", linewidth=2)
axes[0].set_xlabel("Iteration", fontsize=12)
axes[0].set_ylabel("Validation F1-Score", fontsize=12)
axes[0].set_title("GWO Optimisation Progress", fontsize=14, fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# GRU units vs F1 (coloured by LR)
sc = axes[1].scatter(
    gwo_log_df["gru_units"], gwo_log_df["f1_score"],
    s=100, alpha=0.7, c=gwo_log_df["learning_rate"], cmap="viridis",
)
plt.colorbar(sc, ax=axes[1], label="Learning Rate")
axes[1].set_xlabel("GRU Units", fontsize=12)
axes[1].set_ylabel("Validation F1-Score", fontsize=12)
axes[1].set_title("GRU Units vs F1-Score (coloured by LR)", fontsize=14, fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
gwo_prog_path = f"{DIRS['plots']}/{MODEL_NAME}_gwo_progress.png"
plt.savefig(gwo_prog_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {gwo_prog_path}")


In [ ]:

# Final Summary

print("" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)

print("[Baseline]")
print(f"  Hyperparams : LR={BASELINE_PARAMS['learning_rate']}  Batch={BASELINE_PARAMS['batch_size']}  ", end="")
print(f"GRU={BASELINE_PARAMS['gru_units']}  Drop={BASELINE_PARAMS['dropout']}")
print(f"  Test F1     : {baseline_test_metrics.f1():.4f}")
print(f"  Test Acc    : {baseline_test_metrics.accuracy():.4f}")
print(f"  Saved model : {BASELINE_MODEL_PATH}")
print(f"  Train time  : {str(timedelta(seconds=int(baseline_time)))}")

print("[GWO-Optimized]")
print(f"  Hyperparams : LR={best_params['learning_rate']:.2e}  Batch={best_params['batch_size']}  ", end="")
print(f"GRU={best_params['gru_units']}  Drop={best_params['dropout']:.4f}")
print(f"  Test F1     : {optimized_test_metrics.f1():.4f}")
print(f"  Test Acc    : {optimized_test_metrics.accuracy():.4f}")
print(f"  Saved model : {OPTIMIZED_MODEL_PATH}")
print(f"  Train time  : {str(timedelta(seconds=int(optimized_time)))}")

print(f"[Improvement]")
print(f"  F1  : {f1_improvement:+.2f}%")
print(f"  Acc : {acc_improvement:+.2f}%")
print("=" * 70)
